# The Determinants of Small Shareholder Victory

In [1]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      StataNow 19.5
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2025 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 2-core network, expiring 31 Oct 2026
Serial number: 501909305069
  Licensed to: Yichen Luo
               UCL

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


In [2]:
%%stata

clear all
set more off
set varabbrev off
version 18


* Paths
global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-governance/processed_data"
global TABLES "tables"


. 
. clear all

. set more off

. set varabbrev off

. version 18

. 
. 
. * Paths
. global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-gove
> rnance/processed_data"

. global TABLES "tables"

. 


In [3]:
%%stata

import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear

* Parse date
gen day = date(date, "YMD")
format day %td
gen year = year(day)

* Independent variables
local complexity multi_choices weighted quorum


* Label variables
label var non_whale_victory_vn "\${\it Small Win}_{i,t}\$"
label var weighted "\${\it Weighted}_{i,t}\$"
label var quorum "\${\it Quorum}_{i,t}\$"
label var multi_choices "\${\it Multiple Choices}_{i,t}\$"
label var have_discussion "\${\it Discussion}_{i,t}\$"
label var delegation "\${\it Delegation}_{i,t}\$"
label var non_whale_user      "\${\it User}_{i,t}^{\it Small}\$"
label var concensus_diff "\${\it \Delta Concensus}_{i,t}\$"


. 
. import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear
(encoding automatically selected: ISO-8859-1)
(65 vars, 2,830 obs)

. 
. * Parse date
. gen day = date(date, "YMD")

. format day %td

. gen year = year(day)

. 
. * Independent variables
. local complexity multi_choices weighted quorum

. 
. 
. * Label variables
. label var non_whale_victory_vn "\${\it Small Win}_{i,t}\$"

. label var weighted "\${\it Weighted}_{i,t}\$"

. label var quorum "\${\it Quorum}_{i,t}\$"

. label var multi_choices "\${\it Multiple Choices}_{i,t}\$"

. label var have_discussion "\${\it Discussion}_{i,t}\$"

. label var delegation "\${\it Delegation}_{i,t}\$"

. label var non_whale_user      "\${\it User}_{i,t}^{\it Small}\$"

. label var concensus_diff "\${\it \Delta Concensus}_{i,t}\$"

. 


## Proposal Characteristics

In [4]:
%%stata

******************************************************
* LOGISTIC REGRESSIONS
******************************************************

eststo clear

    ppmlhdfe non_whale_victory_vn `complexity' delegation have_discussion, absorb(year categories) vce(cluster categories)
    eststo baseline
    estadd local fe_proposal "Y"
    estadd local fe_time  "Y"

    ppmlhdfe non_whale_victory_vn `complexity' delegation concensus_diff, absorb(year categories) vce(cluster categories)
    eststo discussion
    estadd local fe_proposal "Y"
    estadd local fe_time  "Y"

    ppmlhdfe non_whale_victory_vn `complexity' delegation have_discussion non_whale_user, absorb(year categories) vce(cluster categories)
    eststo user
    estadd local fe_proposal "Y"
    estadd local fe_time  "Y"

* Export LaTeX table
esttab                                                           ///
    baseline discussion user                                     ///
    using "$TABLES/reg_win.tex", replace                         ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs nomtitles                                   ///
    b(%9.3f) se(%9.2f)                                           ///
    noomitted eqlabels(none)                                     ///
    order(delegation have_discussion concensus_diff non_whale_user multi_choices weighted quorum) ///
    mgroups("\${\it Small Win}_{i,t}\$",                         ///
            span                                                 ///
            pattern(1 0 0)                                       ///
            prefix(\multicolumn{@span}{c}{)                      ///
            suffix(})                                            ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    posthead(\cmidrule(lr){2-4} & \multicolumn{1}{c}{Full} & \multicolumn{1}{c}{Discussion \$\geq\$ 2} & \multicolumn{1}{c}{Have Dapp} \\ \cmidrule(lr){2-4} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} \\ \midrule) ///
    substitute("\_" "_")                                         ///
    stats(fe_proposal fe_time N r2_p,                            ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
         labels("Category FE" "Year FE" "Observations" "Pseudo R²"))


. 
. ******************************************************
. * LOGISTIC REGRESSIONS
. ******************************************************
. 
. eststo clear

. 
.     ppmlhdfe non_whale_victory_vn `complexity' delegation have_discussion, ab
> sorb(year categories) vce(cluster categories)
(dropped 349 observations that are either singletons or separated by a fixed ef
> fect)
(ReLU separation check: maximum number of iterations reached; aborting)
Iteration 1:   deviance = 3.1098e+02  eps = .         iters = 5    tol = 1.0e-0
> 4                                                                            
>    min(eta) =  -3.65  P   
Iteration 2:   deviance = 2.2462e+02  eps = 3.84e-01  iters = 5    tol = 1.0e-0
> 4                                                                            
>    min(eta) =  -4.82      
Iteration 3:   deviance = 2.0637e+02  eps = 8.84e-02  iters = 5    tol = 1.0e-0
> 4                                                                            
>    min(